# YOLO26 Video Inference

Runs a YOLO26 `.pt` model on every frame of an MP4 that is already captured at 1 fps.

**Upload your `.pt` model and `.mp4` video when prompted.**

## Step 1: Install & import

In [ ]:
!pip install -q ultralytics

import cv2
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

## Step 2: Upload model and video

In [ ]:
print('Upload your YOLO26 .pt model file:')
model_upload = files.upload()
model_path = next(k for k in model_upload if k.endswith('.pt'))

print('\nUpload your .mp4 video (already at 1 fps):')
video_upload = files.upload()
video_path = next(k for k in video_upload if k.lower().endswith('.mp4'))

print(f'\nModel: {model_path}')
print(f'Video: {video_path}')

## Step 3: Run inference on every frame

Since the video is already at 1 fps, we process every frame (no sub-sampling).
Annotated frames are written to `output.mp4` and per-detection rows to `detections.csv`.

In [ ]:
model = YOLO(model_path)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height}, {fps:.2f} fps, {total} frames')

out_video = 'output.mp4'
writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

rows = []
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model(frame, verbose=False)[0]
    annotated = results.plot()
    writer.write(annotated)

    if results.boxes is not None and len(results.boxes) > 0:
        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), conf, cls in zip(boxes, confs, clss):
            rows.append({
                'frame': frame_idx,
                't_sec': frame_idx,  # 1 fps -> frame index == seconds
                'class_id': int(cls),
                'class_name': model.names.get(int(cls), str(cls)),
                'conf': float(conf),
                'x1': float(x1), 'y1': float(y1),
                'x2': float(x2), 'y2': float(y2),
            })

    frame_idx += 1
    if frame_idx % 25 == 0:
        print(f'  processed {frame_idx}/{total} frames')

cap.release()
writer.release()

det = pd.DataFrame(rows)
det.to_csv('detections.csv', index=False)
print(f'\nDone. {frame_idx} frames processed, {len(det)} detections written to detections.csv')
det.head()

## Step 4: Download results

In [ ]:
files.download('output.mp4')
files.download('detections.csv')